# Step 2: Download and Clean Light Curves

**Why this step:** Raw light curves are not analysis-ready -- they contain missing data points, extreme outliers from cosmic ray hits, and slow instrumental trends layered on top of the real signal. The distinction between astrophysical signal and systematics only holds if the obvious, well-understood instrumental trends are removed first.

This notebook downloads every available sector's light curve for each target from Step 1, cleans it, resamples it to a fixed length, and caches the result to disk.

**IMPORTANT:** Run this notebook from the SAME folder where you want `lightcurve_cache/` to live -- Step 3 (and later steps) expect to find that folder relative to wherever they're run from.

In [ ]:
import numpy as np
import pandas as pd
import lightkurve as lk
from pathlib import Path

## Config

In [ ]:
INPUT_CSV = "multisector_tic_targets.csv"
# Step 1's output -- the CSV of TIC IDs we're going to download data for.

CACHE_DIR = Path("lightcurve_cache")
CACHE_DIR.mkdir(exist_ok=True)
# CACHE_DIR is where every cleaned, resampled light curve gets saved.
# .mkdir(exist_ok=True) actually creates this folder on disk if it
# doesn't already exist.

N_RESAMPLE_POINTS = 2000
# Every light curve gets resampled to exactly this many points, no
# matter how many points it originally had. The VAE (Step 4) needs
# every input to be the SAME fixed size.

FLATTEN_WINDOW_LENGTH = 401
# Controls how "zoomed out" flatten() is when estimating and removing
# the slow, long-term instrumental trend. Must be odd (the underlying
# filter needs a symmetric window with one center point).

SIGMA_CLIP_THRESHOLD = 5
# How aggressive outlier removal is. Any point more than 5 standard
# deviations from the mean gets removed as a likely cosmic-ray hit or
# instrumental spike.

MAX_GAP_FOR_INTERP = 0.5  # days
# When resampling onto the fixed grid, points that don't land on a real
# observation get their value INTERPOLATED. If the nearest real
# observation is more than 0.5 days away, we flag that point as
# untrustworthy instead of trusting the interpolated guess.

## Step functions

In [ ]:
def get_cache_path(tic_id, sector):
    """Where a cleaned, resampled light curve for a given (star, sector) lives on disk."""
    return CACHE_DIR / f"TIC{tic_id}_sector{sector}.npy"

In [ ]:
def download_sectors(tic_id):
    """Search MAST for all SPOC light curves for this TIC ID and download them."""
    search_result = lk.search_lightcurve(
        f"TIC {tic_id}",
        mission="TESS",
        author="SPOC",
        # SPOC = TESS's primary, most rigorously validated pipeline --
        # keeps every target's starting point consistent.
    )

    if len(search_result) == 0:
        # No SPOC light curves exist for this star -- not an error,
        # just an expected outcome for some targets.
        return None

    # Downloads the actual FITS files (or uses lightkurve's own local
    # cache) and returns one LightCurve object per matched sector.
    lc_collection = search_result.download_all()
    return lc_collection

In [ ]:
def clean_light_curve(lc):
    """
    Remove NaNs, normalize, clip outliers, and flatten long-term trends.
    ORDER MATTERS -- each step assumes the previous one is already done.
    """
    # 1. Remove missing/invalid flux values first -- NaNs would corrupt
    #    every calculation in the steps below.
    lc = lc.remove_nans()

    # 2. Normalize flux to a relative scale (~1.0) -- puts every star on
    #    equal footing so a fixed sigma-threshold means the same thing
    #    for every star.
    lc = lc.normalize()

    # 3. Remove extreme outliers (cosmic ray hits etc.) BEFORE
    #    flattening, so they don't distort flatten()'s trend-fit.
    lc = lc.remove_outliers(sigma=SIGMA_CLIP_THRESHOLD)

    # 4. Flatten out slow instrumental drift last, now that the data is
    #    NaN-free, normalized, and outlier-free.
    lc = lc.flatten(window_length=FLATTEN_WINDOW_LENGTH)

    return lc

In [ ]:
def resample_light_curve(lc, n_points=N_RESAMPLE_POINTS, max_gap_days=MAX_GAP_FOR_INTERP):
    """Resample onto a fixed-length uniform time grid, and flag interpolated gap regions."""

    # Strip astropy's units/metadata wrapper, leaving plain numpy arrays.
    time = lc.time.value
    flux = lc.flux.value

    # Build the new FIXED, evenly-spaced time grid every light curve
    # will be measured against.
    t_min, t_max = time.min(), time.max()
    uniform_time = np.linspace(t_min, t_max, n_points)

    # For every point on the new grid, estimate a flux value by drawing
    # a straight line between the two nearest REAL observed points.
    resampled_flux = np.interp(uniform_time, time, flux)

    # Start by assuming every point on the new grid is trustworthy.
    gap_mask = np.ones(n_points, dtype=bool)

    # Check each new grid point individually: how far away is the
    # nearest REAL observation? If farther than max_gap_days, this
    # point's interpolated value is just a guess sitting inside a
    # genuine data gap -- mark it untrustworthy (False).
    for i, t in enumerate(uniform_time):
        nearest_real_gap = np.min(np.abs(time - t))
        if nearest_real_gap > max_gap_days:
            gap_mask[i] = False

    return uniform_time, resampled_flux, gap_mask

In [ ]:
def process_target(tic_id):
    """Full pipeline for a single target: download -> clean -> resample -> cache each sector."""
    print(f"Processing TIC {tic_id}...")

    lc_collection = download_sectors(tic_id)
    if lc_collection is None:
        print(f"  No SPOC light curves found for TIC {tic_id}, skipping.")
        return

    for lc in lc_collection:
        # lc.meta is a dictionary of this light curve's FITS header info.
        # .get("SECTOR", "unknown") safely looks up the sector number.
        sector = lc.meta.get("SECTOR", "unknown")
        cache_path = get_cache_path(tic_id, sector)

        # CACHING CHECK: if this exact star-sector combo was already
        # processed in a previous run, skip it entirely -- avoids
        # re-downloading FITS files from MAST every time we re-run.
        if cache_path.exists():
            print(f"  Sector {sector} already cached, skipping.")
            continue

        # try/except so one problematic sector doesn't crash the whole
        # multi-target run.
        try:
            cleaned = clean_light_curve(lc)
            uniform_time, resampled_flux, gap_mask = resample_light_curve(cleaned)

            # allow_pickle=True is required because we're saving a
            # dictionary (a general Python object), not a plain array.
            np.save(
                cache_path,
                {
                    "time": uniform_time,
                    "flux": resampled_flux,
                    "gap_mask": gap_mask,
                    "tic_id": tic_id,
                    "sector": sector,
                },
                allow_pickle=True,
            )
            print(f"  Sector {sector} cleaned and cached -> {cache_path.name}")

        except Exception as e:
            print(f"  Sector {sector} FAILED ({e}), skipping.")
            continue

## Run

In [ ]:
# Load the target list produced by Step 1.
targets = pd.read_csv(INPUT_CSV)

# OPTIONAL: use a small subset first to test the pipeline / estimate
# timing before committing to the full target list.
# targets = targets.head(10)

print(f"Processing {len(targets)} targets, total {targets['n_sectors'].sum()} sectors")

In [ ]:
for tic_id in targets["tic_id"]:
    process_target(tic_id)

# CACHE_DIR.resolve() turns the relative folder path into a full
# absolute path, so you know exactly where your cached files ended up.
print("\nDone. Cached files are in:", CACHE_DIR.resolve())

## Sanity check -- inspect one cached file

In [ ]:
import matplotlib.pyplot as plt

# Grab any one cached file to verify cleaning + gap-masking worked
sample_file = next(CACHE_DIR.glob("*.npy"))
data = np.load(sample_file, allow_pickle=True).item()

plt.figure(figsize=(10, 4))
plt.plot(data["time"], data["flux"], color="steelblue", linewidth=0.8, label="flux")
plt.scatter(data["time"][~data["gap_mask"]], data["flux"][~data["gap_mask"]],
            color="red", s=8, label="untrustworthy (gap-filled)")
plt.xlabel("Time (days)")
plt.ylabel("Normalized Flux")
plt.title(f"TIC {data['tic_id']} - Sector {data['sector']}")
plt.legend()
plt.show()